# 01. 동결 probe vs LoRA vs 전체 파인튜닝 — ESM-2로 변이 효과 예측하기

**등급**: 🟢 Mac 로컬 (MPS) · Colab T4에서도 실행 확인(약 12분; `peft`·`Bio` 설치, `torchao` 충돌 시 제거) · **예상 시간**: 약 23분 (대부분 파인튜닝 실행; `FAST=True`로 줄일 수 있다) · **다운로드**: 없음 (모듈 02의 ESM-2 8M·35M, ProteinGym v1.3 캐시를 쓴다) · **설치**: `peft` (LoRA)

**질문**
1. 사전학습된 ESM-2를 **파인튜닝**(LoRA, 전체)하면, 고정된 임베딩 위의 **선형 probe**나 **단순 baseline**보다 변이 효과를 더 잘 예측하는가?
2. 그 답은 **분할**(random vs 위치 블록)과 **assay 유형**(안정성 / 생육 / 활성)에 따라 어떻게 달라지는가?
3. LoRA는 학습 매개변수·시간을 얼마나 줄이고, 전체 파인튜닝과 성능이 같은가?

**설계 (결과를 보기 전에 정한 것)**
- 데이터: ProteinGym v1.3 치환 DMS 중 **단일 치환만**, 길이 ≤ 87인 assay 3종(길이가 같은 변이 서열이라 패딩이 없다): `DN7A_SACS2_Tsuboyama_2023_1JIC`(안정성; 모듈 02와 같음), `IF1_ECOLI_Kelsic_2016`(생육/적합도), `A0A247D711_LISMN_Stadelmann_2021`(활성; 유세포분석).
- 모델: ESM-2 8M(주), 35M(DN7A 한 assay에서 규모 확인). 출력 = 변이 위치(잔기 p ↔ 토큰 p)의 마지막 층 은닉 상태 → 선형 head 1개. 손실 = 학습 y를 표준화한 MSE.
- 비교 5종: **B1**(wt·mutant 원-핫 + BLOSUM62, 선형) · **B2**(B1 + 위치 원-핫) · **probe**(동결 ESM-2 변이 위치 임베딩 + Ridge, 규제는 학습 안에서 그룹 CV) · **LoRA**(q·v, r=8, α=16, dropout 0.05, head 포함 학습) · **전체 파인튜닝**.
- **하이퍼파라미터는 미리 고정**: 5 에폭, 배치 32, AdamW(wd 0.01), 10% 워밍업 후 선형 감소. LR = head 1e-3 · LoRA 1e-3 · 전체 5e-5(backbone). test로 고르지 않았고, 민감도는 §6에서 따로 본다. 검증 세트로 조기 종료하지 않는다.
- 분할: **random**(변이 단위 20% test) vs **position-block**(연속 10개 위치 블록 단위 20% test); 시드 0~2마다 분할과 초기화가 모두 바뀐다. 지표 = test Spearman.
- **주 비교(사전 지정)**: 위치 블록 분할에서 (a) LoRA − probe, (b) 전체 − LoRA. 시드가 3개뿐이라 **구간을 계산하지 않고** 시드별 값과 이긴 횟수를 보고한다.

> 파인튜닝은 학습 데이터가 적을수록(여기서 학습 변이 약 800~1,300개) 과적합하기 쉽고, 고정된 에폭 수·LR이 최적이 아닐 수 있다. 결과는 "이 고정 설정에서의 비교"다.

In [ ]:
import gc
import importlib.metadata as md
import os
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from Bio.Align import substitution_matrices
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit, ShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "SETUP.md").exists():
            return q
    return None


ROOT = find_root()
CACHE = (ROOT / ".cache") if ROOT is not None else Path("cache")
os.environ.setdefault("HF_HOME", str(CACHE / "hf"))
PG = CACHE / "proteingym"

import torch
import torch.nn as nn
from huggingface_hub import snapshot_download
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, EsmModel
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
SEED = 0
FAST = False                       # True: 시드 1개, 에폭 2 (코드 점검용)
print("device:", DEVICE, "| cache:", CACHE, "| FAST:", FAST)
print({k: md.version(k) for k in ["numpy", "pandas", "scikit-learn", "torch", "transformers", "peft"]})

## 1. 데이터와 모델 준비
ProteinGym 파일은 모듈 02에서 받은 캐시를 쓴다(없으면 모듈 02 노트북 01의 내려받기 셀을 먼저 실행). ESM-2는 리비전을 고정해 로컬 스냅샷 경로에서 불러온다(모듈 05에서 본 것처럼 저장소 ID로 열면 고정하지 않은 파일이 섞일 수 있다).

In [ ]:
ESM = {"8M": ("facebook/esm2_t6_8M_UR50D", "c731040fcd8d73dceaa04b0a8e6329b345b0f5df"), "35M": ("facebook/esm2_t12_35M_UR50D", "6fbf070e65b0b7291e7bbcd451118c216cff79d8")}
ESM_DIR = {}
for k, (repo, rev) in ESM.items():
    ESM_DIR[k] = Path(snapshot_download(repo, revision=rev, allow_patterns=["model.safetensors", "*.json", "vocab.txt"]))
TOK = AutoTokenizer.from_pretrained(ESM_DIR["8M"])

ref = pd.read_csv(PG / "DMS_substitutions.csv")
ZIP = PG / "DMS_ProteinGym_substitutions.zip"
ASSAYS = ["DN7A_SACS2_Tsuboyama_2023_1JIC", "IF1_ECOLI_Kelsic_2016", "A0A247D711_LISMN_Stadelmann_2021"]
AA = "ACDEFGHIKLMNPQRSTVWY"


def load_assay(dms_id):
    row = ref[ref.DMS_id == dms_id].iloc[0]
    with zipfile.ZipFile(ZIP) as z:
        df = pd.read_csv(z.open(f"DMS_ProteinGym_substitutions/{row.DMS_filename}"))
    df = df[~df.mutant.str.contains(":")].copy()
    df["wt"], df["pos"], df["mt"] = df.mutant.str[0], df.mutant.str[1:-1].astype(int), df.mutant.str[-1]
    assert all(row.target_seq[p - 1] == w for p, w in zip(df.pos, df.wt)), "wt 잔기가 target_seq와 다름"
    assert df.mutated_sequence.str.len().nunique() == 1, "길이가 같지 않은 서열이 있다"
    return row.target_seq, df.reset_index(drop=True), row


DATA = {}
for a in ASSAYS:
    wt_seq, df, row = load_assay(a)
    ids = TOK(df.mutated_sequence.tolist(), return_tensors="pt", padding=False)["input_ids"]
    DATA[a] = dict(wt=wt_seq, df=df, ids=ids, pos=torch.tensor(df.pos.values), y=df.DMS_score.values.astype(np.float32), L=len(wt_seq), type=row.coarse_selection_type)
    print(f"{a:36s} L={len(wt_seq):3d} singles={len(df):5d} 위치={df.pos.nunique():3d} 유형={row.coarse_selection_type}/{row.selection_type} | y 중앙값 {np.median(DATA[a]['y']):.2f}")

## 2. 분할, baseline, 동결 probe
- 분할과 B1/B2/probe는 모듈 02 노트북 01과 같은 정의다. probe의 특징은 **변이 서열**에서 변이 위치의 마지막 층 은닉 상태(E2).
- Ridge의 규제 세기는 학습 안에서 그룹 CV(분할 종류와 같은 그룹)로 Spearman을 기준으로 고른다. 표준화는 내부 CV의 접힘마다 학습 부분에서만 적합한다(파이프라인).

In [ ]:
BLOSUM = substitution_matrices.load("BLOSUM62")


def onehot(chars):
    M = np.zeros((len(chars), len(AA)))
    M[np.arange(len(chars)), [AA.index(c) for c in chars]] = 1
    return M


def baseline_features(df, L):
    blos = np.array([BLOSUM[w][m] for w, m in zip(df.wt, df.mt)], dtype=float)[:, None]
    B1 = np.hstack([onehot(df.wt), onehot(df.mt), blos])
    P = np.zeros((len(df), L))
    P[np.arange(len(df)), df.pos.values - 1] = 1
    return B1, np.hstack([B1, P])


def make_split(df, kind, seed):
    """반환: (train 인덱스, test 인덱스, 그룹 배열). 시드마다 test가 달라진다."""
    if kind == "random":
        groups = np.arange(len(df))
        tr, te = next(ShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(df))
    else:
        groups = (df.pos.values - 1) // 10
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(df, groups=groups))
    return tr, te, groups


def rho(y, p):
    r = spearmanr(y, p).statistic
    return -1.0 if np.isnan(r) else float(r)


def ridge_eval(X, y, groups, tr, te):
    """표준화 + Ridge를 파이프라인으로 묶어 내부 CV의 접힘마다 학습 부분에서만 표준화한다.
    학습 데이터에서 상수인 열(예: position-block 분할에서 test 위치의 원-핫 열)은 StandardScaler가 스케일 1로 둔다."""
    pipe = make_pipeline(StandardScaler(), Ridge())
    gs = GridSearchCV(pipe, {"ridge__alpha": np.logspace(-1, 5, 13)}, scoring=make_scorer(rho), cv=GroupKFold(n_splits=3).split(X[tr], y[tr], groups[tr]))
    gs.fit(X[tr], y[tr])
    return rho(y[te], gs.predict(X[te]))


@torch.no_grad()
def site_features(size, key, batch=128):
    """동결 ESM-2의 변이 위치 마지막 층 은닉 상태 (N, d)."""
    d = DATA[key]
    model = EsmModel.from_pretrained(ESM_DIR[size], add_pooling_layer=False).eval().to(DEVICE)
    out = []
    for s in range(0, len(d["ids"]), batch):
        ids = d["ids"][s:s + batch].to(DEVICE)
        h = model(input_ids=ids, attention_mask=torch.ones_like(ids)).last_hidden_state
        out.append(h[torch.arange(len(ids)), d["pos"][s:s + batch].to(DEVICE)].float().cpu())
    del model
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    return torch.cat(out).numpy()


FEATS = {(size, a): site_features(size, a) for size in ["8M"] for a in ASSAYS}
FEATS[("35M", ASSAYS[0])] = site_features("35M", ASSAYS[0])
print({k: v.shape for k, v in FEATS.items()})

## 3. 파인튜닝 구현
- `SiteRegressor`: ESM-2 backbone → 변이 위치 은닉 상태 → dropout(0.1) → 선형 head. 변이 서열은 모두 길이가 같아 **패딩이 없다**.
- LoRA(Hu et al. 2021): 가중치 행렬 $W$를 고정하고 $W + \tfrac{\alpha}{r}BA$ ($A\in\mathbb R^{r\times d}, B\in\mathbb R^{d\times r}$, $B$는 0으로 초기화)만 학습한다. 아래 점검에서 **초기 LoRA 모델의 출력이 원본과 정확히 같음**을 확인한다.
- 전체 파인튜닝은 backbone 전부를 낮은 LR(5e-5)로, head는 1e-3으로 학습한다.

In [ ]:
class SiteRegressor(nn.Module):
    def __init__(self, backbone, d):
        super().__init__()
        self.backbone, self.drop, self.head = backbone, nn.Dropout(0.1), nn.Linear(d, 1)

    def forward(self, ids, pos):
        h = self.backbone(input_ids=ids, attention_mask=torch.ones_like(ids)).last_hidden_state
        return self.head(self.drop(h[torch.arange(len(ids), device=ids.device), pos])).squeeze(-1)


def build(size, mode, lora_r=8, lora_targets=("query", "value")):
    backbone = EsmModel.from_pretrained(ESM_DIR[size], add_pooling_layer=False)
    if mode == "lora":
        cfg = LoraConfig(r=lora_r, lora_alpha=2 * lora_r, lora_dropout=0.05, target_modules=list(lora_targets), bias="none")
        backbone = get_peft_model(backbone, cfg)
    return SiteRegressor(backbone, backbone.config.hidden_size if mode != "lora" else backbone.base_model.model.config.hidden_size)


def n_params(net):
    tot = sum(p.numel() for p in net.parameters())
    train = sum(p.numel() for p in net.parameters() if p.requires_grad)
    return train, tot


def finetune(size, mode, key, tr, te, seed, epochs=5, bs=32, lr_head=1e-3, lr_lora=1e-3, lr_full=5e-5, lora_r=8, lora_targets=("query", "value")):
    torch.manual_seed(seed)
    d = DATA[key]
    net = build(size, mode, lora_r, lora_targets).to(DEVICE)
    mu, sd = d["y"][tr].mean(), d["y"][tr].std() + 1e-8
    yz = torch.tensor((d["y"] - mu) / sd)
    if mode == "lora":
        groups = [{"params": [p for n, p in net.backbone.named_parameters() if p.requires_grad], "lr": lr_lora}, {"params": net.head.parameters(), "lr": lr_head}]
    else:
        groups = [{"params": net.backbone.parameters(), "lr": lr_full}, {"params": net.head.parameters(), "lr": lr_head}]
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    steps_per_epoch = int(np.ceil(len(tr) / bs))
    total, warm = steps_per_epoch * epochs, max(1, int(0.1 * steps_per_epoch * epochs))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / (total - warm)))
    g = torch.Generator().manual_seed(seed)
    t0 = time.time()
    net.train()
    last = np.nan
    for ep in range(epochs):
        perm = torch.randperm(len(tr), generator=g).numpy()
        tot = 0.0
        for s in range(0, len(tr), bs):
            ix = tr[perm[s:s + bs]]
            loss = nn.functional.mse_loss(net(d["ids"][ix].to(DEVICE), d["pos"][ix].to(DEVICE)), yz[ix].to(DEVICE))
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            tot += loss.item() * len(ix)
        last = tot / len(tr)
    train_time = time.time() - t0
    net.eval()
    with torch.no_grad():
        pred = torch.cat([net(d["ids"][te[s:s + 256]].to(DEVICE), d["pos"][te[s:s + 256]].to(DEVICE)).float().cpu() for s in range(0, len(te), 256)]).numpy()
    train_p, total_p = n_params(net)
    del net, opt
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    return dict(rho=rho(d["y"][te], pred), train_loss=last, trainable=train_p, total=total_p, seconds=train_time)


# 자체 점검 1: 초기 LoRA 모델은 원본과 같은 출력을 낸다 (B=0)
torch.manual_seed(0)
base = build("8M", "full").to(DEVICE).eval()
lora = build("8M", "lora").to(DEVICE).eval()
lora.head.load_state_dict(base.head.state_dict())
ids0, pos0 = DATA[ASSAYS[0]]["ids"][:16].to(DEVICE), DATA[ASSAYS[0]]["pos"][:16].to(DEVICE)
with torch.no_grad():
    print("초기 LoRA vs 원본 출력의 최대 절대차: %.2e" % (base(ids0, pos0) - lora(ids0, pos0)).abs().max().item())
for name, net in [("전체 파인튜닝", base), ("LoRA q,v r=8", lora)]:
    tp, tot = n_params(net)
    print(f"{name:14s} 학습 매개변수 {tp:>9,d} / 전체 {tot:>9,d} ({100 * tp / tot:.2f}%)")
del base, lora
if DEVICE.type == "mps":
    torch.mps.empty_cache()

## 4. 실행: 3 assay × 2 분할 × 시드 × {B1, B2, probe, LoRA, 전체}
시드마다 분할이 달라지므로 시드 간 편차에는 분할 변동도 들어 있다. 같은 시드에서는 모든 방법이 **같은 train/test**를 쓴다.

In [ ]:
SEEDS = [0] if FAST else [0, 1, 2]
EPOCHS = 2 if FAST else 5
rows = []
t_all = time.time()
for a in ASSAYS:
    d = DATA[a]
    B1, B2 = baseline_features(d["df"], d["L"])
    for kind in ["random", "position-block"]:
        for seed in SEEDS:
            tr, te, groups = make_split(d["df"], kind, seed)
            base_row = dict(assay=a.split("_")[0], split=kind, seed=seed)
            rows.append(dict(base_row, method="B1", rho=ridge_eval(B1, d["y"], groups, tr, te)))
            rows.append(dict(base_row, method="B2", rho=ridge_eval(B2, d["y"], groups, tr, te)))
            rows.append(dict(base_row, method="probe", rho=ridge_eval(FEATS[("8M", a)], d["y"], groups, tr, te)))
            for mode in ["lora", "full"]:
                r = finetune("8M", mode, a, tr, te, seed, epochs=EPOCHS)
                rows.append(dict(base_row, method=mode, **r))
    print(f"{a.split('_')[0]} 완료 (누적 {time.time() - t_all:.0f}초)", flush=True)
RES = pd.DataFrame(rows)

### 결과 — test Spearman (시드 평균 ± 표준편차 [시드별 값])
시드가 3개뿐이라 표준편차는 거칠다. 열 `이긴 시드`는 같은 시드에서 해당 방법이 probe보다 높았던 횟수.

In [ ]:
def cell(sub):
    v = sub.sort_values("seed").rho.values
    return f"{v.mean():.2f}±{v.std():.2f} [" + ", ".join(f"{x:.2f}" for x in v) + "]"


METHODS = ["B1", "B2", "probe", "lora", "full"]
for a in RES.assay.unique():
    print(f"\n=== {a} (8M) ===")
    tab = {}
    for kind in ["random", "position-block"]:
        col = {}
        for m in METHODS:
            sub = RES[(RES.assay == a) & (RES.split == kind) & (RES.method == m)]
            col[m] = cell(sub)
        pr = RES[(RES.assay == a) & (RES.split == kind) & (RES.method == "probe")].sort_values("seed").rho.values
        for m in ["lora", "full"]:
            v = RES[(RES.assay == a) & (RES.split == kind) & (RES.method == m)].sort_values("seed").rho.values
            col[m] += f"  이긴 시드 {int((v > pr).sum())}/{len(v)}"
        tab[kind] = col
    print(pd.DataFrame(tab).to_string())

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
w = 0.38
for ax, a in zip(axs, RES.assay.unique()):
    for j, (kind, c) in enumerate(zip(["random", "position-block"], ["#4c78a8", "#f58518"])):
        m = [RES[(RES.assay == a) & (RES.split == kind) & (RES.method == mm)].rho for mm in METHODS]
        ax.bar(np.arange(len(METHODS)) + (j - 0.5) * w, [x.mean() for x in m], w, yerr=[x.std() for x in m], color=c, label=kind, capsize=2)
        for i, x in enumerate(m):
            ax.scatter(np.full(len(x), i + (j - 0.5) * w), x, s=8, color="k", zorder=3)
    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels(METHODS)
    ax.set_title(a, fontsize=9)
axs[0].set_ylabel("test Spearman")
axs[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## 5. 비용: 학습 매개변수·시간
같은 8M 모델·같은 학습 스텝에서 LoRA와 전체 파인튜닝의 실제 비용. 시간은 Mac MPS 단일 실행값(다른 작업과 겹칠 수 있다)이다.

In [ ]:
cost = RES[RES.method.isin(["lora", "full"])].groupby(["assay", "method"]).agg(학습_매개변수=("trainable", "first"), 전체_매개변수=("total", "first"), 평균_초=("seconds", "mean"), 마지막_에폭_학습손실=("train_loss", "mean")).round(3)
print(cost.to_string())

## 6. 규모와 민감도
**(a) 35M 규모 확인**: DN7A에서만, 시드 0~1, 두 분할. **(b) 민감도**: DN7A·위치 블록·8M·시드 0~1에서 LoRA의 r·대상 층·LR, 전체 파인튜닝의 LR을 바꾼다. 이 표로 설정을 고르지 않는다 — 결론이 사전에 고정한 설정에 얼마나 의존하는지 보는 용도다.

In [ ]:
a = ASSAYS[0]
d = DATA[a]
rows35 = []
t0 = time.time()
for kind in ["random", "position-block"]:
    for seed in ([0] if FAST else [0, 1]):
        tr, te, groups = make_split(d["df"], kind, seed)
        base_row = dict(split=kind, seed=seed)
        rows35.append(dict(base_row, method="probe", rho=ridge_eval(FEATS[("35M", a)], d["y"], groups, tr, te)))
        for mode in ["lora", "full"]:
            rows35.append(dict(base_row, method=mode, **finetune("35M", mode, a, tr, te, seed, epochs=EPOCHS)))
R35 = pd.DataFrame(rows35)
print("35M 확인: %.0f초" % (time.time() - t0))
print(R35.groupby(["split", "method"]).agg(rho_평균=("rho", "mean"), 시드별=("rho", lambda s: ", ".join(f"{x:.2f}" for x in s)), 평균_초=("seconds", "mean")).round(2).to_string())

In [ ]:
configs = {
    "LoRA q,v r=8 lr1e-3 (기본)": dict(mode="lora"),
    "LoRA r=2": dict(mode="lora", lora_r=2),
    "LoRA r=32": dict(mode="lora", lora_r=32),
    "LoRA 전 선형층 r=8": dict(mode="lora", lora_targets=("query", "key", "value", "dense")),      # "dense"는 층마다 attention.output.dense · intermediate.dense · output.dense 세 모듈에 일치
    "LoRA lr 3e-4": dict(mode="lora", lr_lora=3e-4),
    "LoRA lr 3e-3": dict(mode="lora", lr_lora=3e-3),
    "전체 lr 5e-5 (기본)": dict(mode="full"),
    "전체 lr 1e-5": dict(mode="full", lr_full=1e-5),
    "전체 lr 2e-4": dict(mode="full", lr_full=2e-4),
}
d = DATA[ASSAYS[0]]
rows_s = []
t0 = time.time()
for name, kw in configs.items():
    for seed in ([0] if FAST else [0, 1]):
        tr, te, _ = make_split(d["df"], "position-block", seed)
        r = finetune("8M", kw["mode"], ASSAYS[0], tr, te, seed, epochs=EPOCHS, **{k: v for k, v in kw.items() if k != "mode"})
        rows_s.append(dict(설정=name, seed=seed, rho=r["rho"], 학습_매개변수=r["trainable"]))
S = pd.DataFrame(rows_s)
print("민감도: %.0f초" % (time.time() - t0))
print(S.groupby("설정", sort=False).agg(rho_평균=("rho", "mean"), 시드별=("rho", lambda s: ", ".join(f"{x:.2f}" for x in s)), 학습_매개변수=("학습_매개변수", "first")).round(3).to_string())
pr = RES[(RES.assay == "DN7A") & (RES.split == "position-block") & (RES.method == "probe") & (RES.seed.isin([0, 1]))].rho.mean()
print(f"(참고: 같은 시드 0~1의 probe 평균 {pr:.2f})")

## 정리
(실행 결과를 확인한 뒤 README §5에서 수치와 함께 해석한다. 이 노트북은 ESM-2 8M(+35M 한 assay), 짧은 단백질 3개, 시드 3개, 고정된 5 에폭·LR의 결과이며 더 큰 모델·더 많은 데이터·튜닝된 파인튜닝에는 해당하지 않는다.)